In [ ]:
# import libraries
import json
import os
import polars as pl
import polars.selectors as cs
from sqlalchemy import create_engine

In [ ]:
json_path = "../data/raw/foodb_2020_04_07_json/"

# let's open one of these files and see what happens
# ideally, we open each file in the directory
# keep the file name (ie, everything before .json)
# use the file name as a table name in postgres

for filename in os.listdir(json_path):
    if filename.endswith(".json"):
        print(filename)

FoodTaxonomy.json
OntologySynonym.json
NcbiTaxonomyMap.json
Food.json
Enzyme.json
Nutrient.json
Reference.json
Flavor.json
OntologyTerm.json
Pathway.json
Content.json
HealthEffect.json


In [ ]:
for filename in os.listdir(json_path):
    if filename.endswith(".json"):
        df_label = filename[:-5]
        date_columns = ["created_at", "updated_at"]
        print(f"Processing {os.path.join(json_path, filename)}")
        df = pl.read_ndjson(os.path.join(json_path, filename))
        df = df.with_columns(
            cs.by_name(date_columns, require_all=False).cast(pl.Datetime)
        )
        with pl.Config(fmt_str_lengths=1000, tbl_width_chars=1000):
            print(df.head())

Processing ../data/raw/foodb_2020_04_07_json/FoodTaxonomy.json
shape: (5, 7)
┌─────┬─────────┬──────────────────┬─────────────────────┬──────────────────────┬─────────────────────┬─────────────────────┐
│ id  ┆ food_id ┆ ncbi_taxonomy_id ┆ classification_name ┆ classification_order ┆ created_at          ┆ updated_at          │
│ --- ┆ ---     ┆ ---              ┆ ---                 ┆ ---                  ┆ ---                 ┆ ---                 │
│ i64 ┆ i64     ┆ i64              ┆ str                 ┆ i64                  ┆ datetime[μs]        ┆ datetime[μs]        │
╞═════╪═════════╪══════════════════╪═════════════════════╪══════════════════════╪═════════════════════╪═════════════════════╡
│ 1   ┆ 1       ┆ 357850           ┆ "Eukaryota"         ┆ 1                    ┆ 2017-03-29 18:35:53 ┆ 2017-03-29 18:35:53 │
│ 2   ┆ 1       ┆ 357850           ┆ "Viridiplantae"     ┆ 2                    ┆ 2017-03-29 18:35:53 ┆ 2017-03-29 18:35:53 │
│ 3   ┆ 1       ┆ 357850           ┆ "Str

These tables were too wide even with the wider settings
OntologySynonym
Food
Enzyme
Nutrient
Reference
OntologyTerm
Content
HealthEffect

Other fields, need to check for data types
external_id
comment
curator
creator_id
updater_id

In [ ]:
# instead of getting df.head(), can i just print the columns and their data types?
# maybe `glimpse`

for filename in os.listdir(json_path):
    if filename.endswith(".json"):
        df_label = filename[:-5]
        date_columns = ["created_at", "updated_at"]
        print(f"Processing {os.path.join(json_path, filename)}")
        df = pl.read_ndjson(os.path.join(json_path, filename))
        df = df.with_columns(
            cs.by_name(date_columns, require_all=False).cast(pl.Datetime)
        )
        print(df.glimpse())

Processing ../data/raw/foodb_2020_04_07_json/FoodTaxonomy.json
Rows: 889
Columns: 7
$ id                            <i64> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
$ food_id                       <i64> 1, 1, 1, 1, 1, 1, 1, 1, 1, 1
$ ncbi_taxonomy_id              <i64> 357850, 357850, 357850, 357850, 357850, 357850, 357850, 357850, 357850, 357850
$ classification_name           <str> '"Eukaryota"', '"Viridiplantae"', '"Streptophyta"', '"Embryophyta"', '"Tracheophyta"', '"Spermatophyta"', '"Magnoliophyta"', '"eudicotyledons"', '"Gunneridae"', '"Pentapetalae"'
$ classification_order          <i64> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
$ created_at           <datetime[μs]> 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53
$ updated_at           <datetime[μs]> 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 18:35:53, 2017-03-29 

In [ ]:
for filename in os.listdir(json_path):
    if filename.endswith(".json"):
        df_label = filename[:-5]
        date_columns = ["created_at", "updated_at"]
        print(f"Processing {os.path.join(json_path, filename)}")
        df = pl.read_ndjson(os.path.join(json_path, filename))
        df = df.with_columns(
            cs.by_name(date_columns, require_all=False).cast(pl.Datetime)
        )
        print(df.describe())

Processing ../data/raw/foodb_2020_04_07_json/FoodTaxonomy.json
shape: (9, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ statistic  ┆ id         ┆ food_id    ┆ ncbi_taxo ┆ classific ┆ classific ┆ created_a ┆ updated_a │
│ ---        ┆ ---        ┆ ---        ┆ nomy_id   ┆ ation_nam ┆ ation_ord ┆ t         ┆ t         │
│ str        ┆ f64        ┆ f64        ┆ ---       ┆ e         ┆ er        ┆ ---       ┆ ---       │
│            ┆            ┆            ┆ f64       ┆ ---       ┆ ---       ┆ str       ┆ str       │
│            ┆            ┆            ┆           ┆ str       ┆ f64       ┆           ┆           │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ count      ┆ 889.0      ┆ 889.0      ┆ 889.0     ┆ 889       ┆ 889.0     ┆ 889       ┆ 889       │
│ null_count ┆ 0.0        ┆ 0.0        ┆ 0.0       ┆ 0         ┆ 0.0       ┆ 0         ┆ 0         │
│ mean       ┆

We can start with only a few tables first:

- Food
- Nutrient
- Content
- HealthEffect
- Flavor
- FoodTaxonomy

Clean up these tables/types, send to postgres

Can I make a graph DB that has each food item and some similarity measure against all other food items? Maybe weighed by ingredient/compound concentration?

- Ingredient similarity based on compounds (Food, similarity graph)
- Food CONTAINS ingredient (Content, knowledge graph)
- Food CATEGORY compound? (Flavor, knowledge graph)

Build out hierarchy

In [ ]:
food_df = pl.read_ndjson(json_path + "Food.json")

food_df = food_df.with_columns(
    cs.by_name(date_columns, require_all=False).cast(pl.Datetime)
)

food_df.glimpse()

Rows: 992
Columns: 23
$ id                            <i64> 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
$ name                          <str> 'Angelica', 'Savoy cabbage', 'Silver linden', 'Kiwi', 'Allium', 'Garden onion', 'Leek', 'Garlic', 'Chives', 'Lemon verbena'
$ name_scientific               <str> 'Angelica keiskei', 'Brassica oleracea var. sabauda', 'Tilia argentea', 'Actinidia chinensis', 'Allium', 'Allium cepa', 'Allium porrum', 'Allium sativum', 'Allium schoenoprasum', 'Aloysia triphylla'
$ description                   <str> 'Angelica is a genus of about 60 species of tall biennial and perennial herbs in the family Apiaceae, native to temperate and subarctic regions of the Northern Hemisphere, reaching as far north as Iceland and Lapland. They grow to 1–3 m tall, with large bipinnate leaves and large compound umbels of white or greenish-white flowers. Some species can be found in purple moor and rush pastures.', 'Savoy cabbage (Brassica oleracea convar. capitata var. sabauda L. ) is a vari

In [ ]:
content_df = pl.read_ndjson(json_path + "Content.json")

content_df = content_df.with_columns(
    cs.by_name(date_columns, require_all=False).cast(pl.Datetime)
)

content_df.head()

id,source_id,source_type,food_id,orig_food_id,orig_food_common_name,orig_food_scientific_name,orig_food_part,orig_source_id,orig_source_name,orig_content,orig_min,orig_max,orig_unit,orig_citation,citation,citation_type,creator_id,updater_id,created_at,updated_at,orig_method,orig_unit_expression,standard_content,preparation_type,export
i64,i64,str,i64,str,str,str,str,str,str,str,str,str,str,null,str,str,null,null,datetime[μs],datetime[μs],null,null,str,str,i64
1,1,"""Nutrient""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""FAT""","""FAT""","""1955.0""","""70.0""","""3840.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""1955.0""","""raw""",0
2,1,"""Nutrient""",6,"""53""","""Onion""","""Allium cepa L. [Liliaceae]""","""Bulb""","""FAT""","""FAT""","""1853.95""","""100.0""","""3607.9""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""1853.95""","""raw""",0
3,1,"""Nutrient""",6,"""53""","""Onion""","""Allium cepa L. [Liliaceae]""","""Leaf""","""FAT""","""FAT""","""4150.0""","""600.0""","""7700.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""4150.0""","""raw""",0
4,1,"""Nutrient""",9,"""55""","""Chives""","""Allium schoenoprasum L. [Lilia…","""Leaf""","""FAT""","""FAT""","""3900.0""","""300.0""","""7500.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""3900.0""","""raw""",0
5,1,"""Nutrient""",11,"""70""","""Cashew""","""Anacardium occidentale L. [Ana…","""Fruit""","""FAT""","""FAT""","""2500.0""","""100.0""","""4900.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""2500.0""","""other""",0


In [ ]:
content_df.filter(pl.col("orig_food_common_name") == "Kiwi")

id,source_id,source_type,food_id,orig_food_id,orig_food_common_name,orig_food_scientific_name,orig_food_part,orig_source_id,orig_source_name,orig_content,orig_min,orig_max,orig_unit,orig_citation,citation,citation_type,creator_id,updater_id,created_at,updated_at,orig_method,orig_unit_expression,standard_content,preparation_type,export
i64,i64,str,i64,str,str,str,str,str,str,str,str,str,str,null,str,str,null,null,datetime[μs],datetime[μs],null,null,str,str,i64
1,1,"""Nutrient""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""FAT""","""FAT""","""1955.0""","""70.0""","""3840.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""1955.0""","""raw""",0
402,2,"""Nutrient""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""PROTEIN""","""PROTEIN""","""3536.5""","""790.0""","""6283.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:17,2020-04-27 16:20:52,null,null,"""3536.5""","""raw""",0
10771,3,"""Nutrient""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""CARBOHYDRATES""","""CARBOHYDRATES""","""51335.0""","""14880.0""","""87790.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:44:08,2020-04-27 16:21:06,null,null,"""51335.0""","""raw""",0
682040,2576,"""Compound""",4,"""156""","""Kiwi""",null,null,"""125""","""(-)-Epicatechin""","""0.285384618""",null,null,"""mg/100 g""",null,"""PHENOL EXPLORER""","""DATABASE""",null,null,2014-11-05 16:54:27,2020-04-27 16:37:06,null,null,"""0.285384618""","""raw""",1
682144,17121,"""Compound""",4,"""156""","""Kiwi""",null,null,"""128""","""(-)-Epicatechin 3-O-gallate""","""0.0""",null,null,"""mg/100 g""",null,"""PHENOL EXPLORER""","""DATABASE""",null,null,2014-11-05 16:54:28,2020-04-27 16:37:06,null,null,"""0.0""","""raw""",1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
687922,15475,"""Compound""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""LUTEIN""","""LUTEIN""","""0.54""","""0.18""","""0.9""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 16:55:37,2020-04-27 16:37:16,null,null,"""0.54""","""raw""",1
687923,15890,"""Compound""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""NEOXANTHIN""","""NEOXANTHIN""",null,null,null,null,null,"""DUKE""","""DATABASE""",null,null,2014-11-05 16:55:37,2018-06-22 07:00:42,null,null,null,"""raw""",1
687924,16258,"""Compound""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""IRON""","""IRON""","""1.65""","""0.3""","""3.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 16:55:37,2020-04-27 16:37:16,null,null,"""1.65""","""raw""",1


In [ ]:
content_df.filter(pl.col("orig_food_common_name") == "Kiwi").glimpse()

Rows: 67
Columns: 26
$ id                                 <i64> 1, 402, 10771, 682040, 682144, 682237, 682330, 682445, 682545, 682582
$ source_id                          <i64> 1, 2, 3, 2576, 17121, 17707, 17709, 2571, 2572, 17712
$ source_type                        <str> 'Nutrient', 'Nutrient', 'Nutrient', 'Compound', 'Compound', 'Compound', 'Compound', 'Compound', 'Compound', 'Compound'
$ food_id                            <i64> 4, 4, 4, 4, 4, 4, 4, 4, 4, 4
$ orig_food_id                       <str> '29', '29', '29', '156', '156', '156', '156', '156', '156', '156'
$ orig_food_common_name              <str> 'Kiwi', 'Kiwi', 'Kiwi', 'Kiwi', 'Kiwi', 'Kiwi', 'Kiwi', 'Kiwi', 'Kiwi', 'Kiwi'
$ orig_food_scientific_name          <str> 'Actinidia chinensis PLANCHON [Actinidiaceae]', 'Actinidia chinensis PLANCHON [Actinidiaceae]', 'Actinidia chinensis PLANCHON [Actinidiaceae]', None, None, None, None, None, None, None
$ orig_food_part                     <str> 'Fruit', 'Fruit', 'Fruit', None, 

In [ ]:
content_df.select(pl.col("orig_unit").value_counts(sort=True))

orig_unit
struct[2]
"{null,4844705}"
"{""mg/100 g"",792002}"
"{""uM"",16441}"
"{""kcal/100 g"",15658}"
"{""IU"",11729}"
…
"{""IU/100 g"",80}"
"{""umol/g "",28}"
"{""umol/g"",16}"


In [ ]:
content_df

id,source_id,source_type,food_id,orig_food_id,orig_food_common_name,orig_food_scientific_name,orig_food_part,orig_source_id,orig_source_name,orig_content,orig_min,orig_max,orig_unit,orig_citation,citation,citation_type,creator_id,updater_id,created_at,updated_at,orig_method,orig_unit_expression,standard_content,preparation_type,export
i64,i64,str,i64,str,str,str,str,str,str,str,str,str,str,null,str,str,null,null,datetime[μs],datetime[μs],null,null,str,str,i64
1,1,"""Nutrient""",4,"""29""","""Kiwi""","""Actinidia chinensis PLANCHON […","""Fruit""","""FAT""","""FAT""","""1955.0""","""70.0""","""3840.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""1955.0""","""raw""",0
2,1,"""Nutrient""",6,"""53""","""Onion""","""Allium cepa L. [Liliaceae]""","""Bulb""","""FAT""","""FAT""","""1853.95""","""100.0""","""3607.9""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""1853.95""","""raw""",0
3,1,"""Nutrient""",6,"""53""","""Onion""","""Allium cepa L. [Liliaceae]""","""Leaf""","""FAT""","""FAT""","""4150.0""","""600.0""","""7700.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""4150.0""","""raw""",0
4,1,"""Nutrient""",9,"""55""","""Chives""","""Allium schoenoprasum L. [Lilia…","""Leaf""","""FAT""","""FAT""","""3900.0""","""300.0""","""7500.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""3900.0""","""raw""",0
5,1,"""Nutrient""",11,"""70""","""Cashew""","""Anacardium occidentale L. [Ana…","""Fruit""","""FAT""","""FAT""","""2500.0""","""100.0""","""4900.0""","""mg/100 g""",null,"""DUKE""","""DATABASE""",null,null,2014-11-05 13:42:11,2020-04-27 16:20:52,null,null,"""2500.0""","""other""",0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
6180503,30807,"""Compound""",268,null,"""Jiaogulan beer""",null,null,null,null,"""1.08""","""0.0""","""0.0""",null,null,"""Zhu Yan, Zhang Xingde, and Niu…","""ARTICLE""",null,null,2020-04-29 22:59:30,2020-04-29 22:59:30,null,null,"""1.08""","""beverage, alcoholic""",1
6180504,31057,"""Compound""",268,null,"""Guanshan beer""",null,null,null,null,"""0.27""","""0.0""","""0.0""",null,null,"""Zhu Yan, Zhang Xingde, and Niu…","""ARTICLE""",null,null,2020-04-29 22:59:30,2020-04-29 22:59:30,null,null,"""0.27""","""beverage, alcoholic""",1
6180505,31057,"""Compound""",268,null,"""Jiaogulan beer""",null,null,null,null,"""0.54""","""0.0""","""0.0""",null,null,"""Zhu Yan, Zhang Xingde, and Niu…","""ARTICLE""",null,null,2020-04-29 22:59:30,2020-04-29 22:59:30,null,null,"""0.54""","""beverage, alcoholic""",1


In [ ]:
content_df.filter(
    pl.col("orig_source_id").is_not_null() & pl.col("orig_source_name").is_not_null()
).group_by("orig_food_common_name", maintain_order=True).agg(
    components=pl.struct(
        component_id="orig_source_id",
        component="orig_source_name",
        amount="orig_content",
        unit="orig_unit",
    )
)

orig_food_common_name,components
str,list[struct[4]]
"""Kiwi""","[{""FAT"",""FAT"",""1955.0"",""mg/100 g""}, {""PROTEIN"",""PROTEIN"",""3536.5"",""mg/100 g""}, … {""PECTIN"",""PECTIN"",""1450.0"",""mg/100 g""}]"
"""Onion""","[{""FAT"",""FAT"",""1853.95"",""mg/100 g""}, {""FAT"",""FAT"",""4150.0"",""mg/100 g""}, … {""9,10,13-TRIHYDROXY-OCTADEC-11-ENOIC-ACID"",""9,10,13-TRIHYDROXY-OCTADEC-11-ENOIC-ACID"",null,null}]"
"""Chives""","[{""FAT"",""FAT"",""3900.0"",""mg/100 g""}, {""PROTEIN"",""PROTEIN"",""18400.0"",""mg/100 g""}, … {""ISORHAMNETIN-3-BETA-D-GLUCOSIDE"",""ISORHAMNETIN-3-BETA-D-GLUCOSIDE"",null,null}]"
"""Cashew""","[{""FAT"",""FAT"",""2500.0"",""mg/100 g""}, {""FAT"",""FAT"",""1300.0"",""mg/100 g""}, … {""GLUCURONIC-ACID"",""GLUCURONIC-ACID"",null,null}]"
"""Pineapple""","[{""FAT"",""FAT"",""2188.6"",""mg/100 g""}, {""PROTEIN"",""PROTEIN"",""2950.0"",""mg/100 g""}, … {""PECTIN"",""PECTIN"",""110.0"",""mg/100 g""}]"
…,…
"""Rose Hips""","[{""SUCROSE"",""SUCROSE"",null,null}, {""ASCORBIC-ACID"",""ASCORBIC-ACID"",""675.0"",""mg/100 g""}, … {""PECTINS"",""PECTINS"",""4000.0"",""mg/100 g""}]"
"""Salmonberry""","[{""VANILLIC-ACID"",""VANILLIC-ACID"",null,null}, {""PROTOCATECHUIC-ACID"",""PROTOCATECHUIC-ACID"",null,null}, … {""4-HYDROXY-BENZOIC-ACID"",""4-HYDROXY-BENZOIC-ACID"",null,null}]"
"""Japanese Horseradish""","[{""GLUCOPUTRANJIVIN"",""GLUCOPUTRANJIVIN"",null,null}, {""SINIGRIN"",""SINIGRIN"",null,null}, … {""GLUCOCAPPARIN"",""GLUCOCAPPARIN"",null,null}]"


It's cool to get the struct out with the id name, common name, amount/concentration, and units, but these seem incomplete.

Go back to only using the lists of common names

In [ ]:
content_df.filter(
    pl.col("orig_source_id").is_not_null() & pl.col("orig_source_name").is_not_null()
).select(pl.col("orig_unit").value_counts(sort=True))

orig_unit
struct[2]
"{""mg/100 g"",630090}"
"{null,30091}"
"{""kcal/100 g"",15658}"
"{""IU"",11725}"
"{""RE"",7307}"
"{""α-TE"",1055}"
"{""NE"",1055}"


In [ ]:
content_df.filter(
    pl.col("orig_source_id").is_not_null() & pl.col("orig_source_name").is_not_null()
).group_by("orig_food_common_name", maintain_order=True).agg(
    pl.col("orig_source_name")
).with_columns(
    pl.col("orig_source_name").list.join("||")
).with_columns(
    pl.col("orig_source_name").str.split("||").alias("components")
)

orig_food_common_name,orig_source_name,components
str,str,list[str]
"""Kiwi""","""FAT||PROTEIN||CARBOHYDRATES||(…","[""FAT"", ""PROTEIN"", … ""PECTIN""]"
"""Onion""","""FAT||FAT||PROTEIN||PROTEIN||CA…","[""FAT"", ""FAT"", … ""9,10,13-TRIHYDROXY-OCTADEC-11-ENOIC-ACID""]"
"""Chives""","""FAT||PROTEIN||CARBOHYDRATES||T…","[""FAT"", ""PROTEIN"", … ""ISORHAMNETIN-3-BETA-D-GLUCOSIDE""]"
"""Cashew""","""FAT||FAT||FAT||PROTEIN||PROTEI…","[""FAT"", ""FAT"", … ""GLUCURONIC-ACID""]"
"""Pineapple""","""FAT||PROTEIN||CARBOHYDRATES||(…","[""FAT"", ""PROTEIN"", … ""PECTIN""]"
…,…,…
"""Rose Hips""","""SUCROSE||ASCORBIC-ACID||TILIRO…","[""SUCROSE"", ""ASCORBIC-ACID"", … ""PECTINS""]"
"""Salmonberry""","""VANILLIC-ACID||PROTOCATECHUIC-…","[""VANILLIC-ACID"", ""PROTOCATECHUIC-ACID"", … ""4-HYDROXY-BENZOIC-ACID""]"
"""Japanese Horseradish""","""GLUCOPUTRANJIVIN||SINIGRIN||AL…","[""GLUCOPUTRANJIVIN"", ""SINIGRIN"", … ""GLUCOCAPPARIN""]"


In [ ]:
import 

In [ ]:
epic_df = pl.read_parquet("../data/processed/cleaned_df.parquet.gzip")

epic_df.head(3)

In [ ]:
# connect to MySQL

mariadb_key_path = "../../../../secrets/pw.json"

with open(mariadb_key_path, "r") as fo:
    mariadb_key = json.loads(fo.read())
user = mariadb_key["user"]
password = mariadb_key["password"]
host = "192.168.1.24"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}/scraped_recipes?charset=utf8"
)

In [ ]:
with engine.begin() as connection:
    df = pd.read_sql_table(
        table_name="all_recipes_world_cuisines",
        con=connection,
        columns=[
            "id",
            "recipe_title",
            "recipe_url",
            "recipe_photo",
            "description",
            "ingredients",
            "steps",
            "cuisine",
        ],
    )

df